### Concurrency Models 

#### URL Fetcher 

##### Threaded version


In [4]:
import threading
import requests
import time

URL = "https://httpbin.org/delay/2"
NUM_REQUESTS = 10

def fetch():
    response = requests.get(URL)
    return response.status_code

start = time.time()

threads = []
for _ in range(NUM_REQUESTS):
    t = threading.Thread(target=fetch)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("Threading Time:", time.time() - start)

Threading Time: 4.156283378601074


##### Multiprocessing Version

In [5]:
from multiprocessing import Process
import requests
import time

URL = "https://httpbin.org/delay/2"
NUM_REQUESTS = 10

def fetch():
    response = requests.get(URL)
    return response.status_code

start = time.time()

processes = []
for _ in range(NUM_REQUESTS):
    p = Process(target=fetch)
    p.start()
    processes.append(p)

for p in processes:
    p.join()

print("Multiprocessing Time:", time.time() - start)

Multiprocessing Time: 0.4788017272949219


##### Asyncio Version

In [9]:
import asyncio
import aiohttp
import time

URL = "https://httpbin.org/delay/2"
NUM_REQUESTS = 10

async def fetch(session):
    async with session.get(URL) as response:
        return response.status

async def main():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session) for _ in range(NUM_REQUESTS)]
        await asyncio.gather(*tasks)

start = time.time()
# asyncio.run(main())
await main()
print("Asyncio Time:", time.time() - start)

Asyncio Time: 3.394045829772949


### Async Web Scrapper

In [10]:
import asyncio
import aiohttp
import time

URLS = [
    "https://httpbin.org/delay/2" for _ in range(10)
]

async def fetch(session, url, index):
    try:
        print(f"Starting request {index}")
        
        async with session.get(url) as response:
            text = await response.text()
            
            print(f"Finished request {index}")
            return len(text)
    
    except Exception as e:
        print(f"Error in request {index}: {e}")
        return None


async def main():
    start = time.time()
    
    async with aiohttp.ClientSession() as session:
        tasks = [
            fetch(session, url, i)
            for i, url in enumerate(URLS)
        ]
        
        results = await asyncio.gather(*tasks)
    
    print("\nAll done!")
    print("Results:", results)
    print("Total time:", time.time() - start)


# asyncio.run(main())
await main()

Starting request 0
Starting request 1
Starting request 2
Starting request 3
Starting request 4
Starting request 5
Starting request 6
Starting request 7
Starting request 8
Starting request 9
Finished request 8
Finished request 4
Finished request 5
Finished request 1
Finished request 7
Finished request 6
Finished request 3
Finished request 9
Finished request 0
Finished request 2

All done!
Results: [368, 368, 368, 368, 368, 368, 368, 368, 368, 368]
Total time: 3.1914048194885254


### Async File Processor

In [12]:
# create dummy files for testing
for i in range(10):
    with open(f"file_{i}.txt", "w") as f:
        f.write("Hello async world\n" * 100000)

In [13]:
import asyncio
import aiofiles
import time

FILES = [f"file_{i}.txt" for i in range(10)]

async def process_file(filename):
    print(f"Reading {filename}")
    
    async with aiofiles.open(filename, mode="r") as f:
        content = await f.read()
    
    # Simulate processing
    word_count = len(content.split())
    
    print(f"Finished {filename} — Words: {word_count}")
    return word_count


async def main():
    start = time.time()
    
    tasks = [process_file(file) for file in FILES]
    
    results = await asyncio.gather(*tasks)
    
    print("\nAll files processed.")
    print("Total words:", sum(results))
    print("Total time:", time.time() - start)


# asyncio.run(main())
await main()

Reading file_0.txt
Reading file_1.txt
Reading file_2.txt
Reading file_3.txt
Reading file_4.txt
Reading file_5.txt
Reading file_6.txt
Reading file_7.txt
Reading file_8.txt
Reading file_9.txt
Finished file_0.txt — Words: 300000
Finished file_5.txt — Words: 300000
Finished file_2.txt — Words: 300000
Finished file_4.txt — Words: 300000
Finished file_6.txt — Words: 300000
Finished file_3.txt — Words: 300000
Finished file_1.txt — Words: 300000
Finished file_8.txt — Words: 300000
Finished file_7.txt — Words: 300000
Finished file_9.txt — Words: 300000

All files processed.
Total words: 3000000
Total time: 0.7338709831237793


### Implement retry logic with exponential backoff for async HTTP calls

In [21]:
import asyncio
import aiohttp
import random

URL = "https://httpbin.org/status/500"

async def fetch_with_retry(session, url, retries=5, base_delay=1):
    for attempt in range(retries):
        try:
            print(f"Attempt {attempt+1}")
            
            async with session.get(url) as response:
                response.raise_for_status()
                return await response.text()

        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            if attempt == retries - 1:
                print("All retries failed.")
                raise
            
            delay = base_delay * (2 ** attempt)
            jitter = random.uniform(0, 0.5)
            
            print(f"Error: {e}. Retrying in {delay + jitter:.2f}s...")
            
            await asyncio.sleep(delay + jitter)


async def main():
    timeout = aiohttp.ClientTimeout(total=3)

    async with aiohttp.ClientSession(timeout=timeout) as session:
        try:
            result = await fetch_with_retry(session, URL)
            print("Success!")
        except Exception as e:
            print("Request ultimately failed:", e)

# asyncio.run(main())
await main()

Attempt 1
Error: 500, message='INTERNAL SERVER ERROR', url='https://httpbin.org/status/500'. Retrying in 1.38s...
Attempt 2
Error: 500, message='INTERNAL SERVER ERROR', url='https://httpbin.org/status/500'. Retrying in 2.03s...
Attempt 3
Error: 500, message='INTERNAL SERVER ERROR', url='https://httpbin.org/status/500'. Retrying in 4.28s...
Attempt 4
Error: 500, message='INTERNAL SERVER ERROR', url='https://httpbin.org/status/500'. Retrying in 8.26s...
Attempt 5
All retries failed.
Request ultimately failed: 500, message='INTERNAL SERVER ERROR', url='https://httpbin.org/status/500'


### Producer-Consumer Pipeline: producer generates tasks, consumers process them with concurrency limit

In [24]:
import asyncio
import random
import time

TOTAL_TASKS = 20
NUM_CONSUMERS = 5
CONCURRENCY_LIMIT = 3


async def producer(queue: asyncio.Queue):
    """Generate tasks and put them into the queue."""
    for task_id in range(TOTAL_TASKS):
        await asyncio.sleep(0.1)  # simulate production delay
        print(f"[Producer] Generated task {task_id}")
        await queue.put(task_id)

    # Send stop signals for consumers
    for _ in range(NUM_CONSUMERS):
        await queue.put(None)

    print("[Producer] Finished producing")


async def process_task(task_id: int):
    """Simulate I/O-bound processing."""
    work_time = random.uniform(0.5, 2)
    print(f"    Processing task {task_id} (takes {work_time:.2f}s)")
    await asyncio.sleep(work_time)
    print(f"    Finished task {task_id}")


async def consumer(queue: asyncio.Queue, semaphore: asyncio.Semaphore, consumer_id: int):
    """Continuously consume tasks from queue."""
    while True:
        task = await queue.get()

        if task is None:
            print(f"[Consumer {consumer_id}] Shutting down")
            queue.task_done()
            break

        async with semaphore:  # concurrency control
            print(f"[Consumer {consumer_id}] Picked task {task}")
            await process_task(task)

        queue.task_done()


async def main():
    queue = asyncio.Queue(maxsize=10)  # backpressure control
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    start = time.time()

    # Start producer
    producer_task = asyncio.create_task(producer(queue))

    # Start consumers
    consumer_tasks = [
        asyncio.create_task(consumer(queue, semaphore, i))
        for i in range(NUM_CONSUMERS)
    ]

    # Wait for producer to finish
    await producer_task

    # Wait until all tasks processed
    await queue.join()

    # Wait for consumers to exit
    await asyncio.gather(*consumer_tasks)

    print(f"\nPipeline completed in {time.time() - start:.2f}s")


# asyncio.run(main())
await main()

[Producer] Generated task 0
[Consumer 0] Picked task 0
    Processing task 0 (takes 1.75s)
[Producer] Generated task 1
[Consumer 1] Picked task 1
    Processing task 1 (takes 1.42s)
[Producer] Generated task 2
[Consumer 2] Picked task 2
    Processing task 2 (takes 1.66s)
[Producer] Generated task 3
[Producer] Generated task 4
[Producer] Generated task 5
[Producer] Generated task 6
[Producer] Generated task 7
[Producer] Generated task 8
[Producer] Generated task 9
[Producer] Generated task 10
[Producer] Generated task 11
[Producer] Generated task 12
[Producer] Generated task 13
[Producer] Generated task 14
    Finished task 1
[Consumer 3] Picked task 3
    Processing task 3 (takes 0.61s)
[Producer] Generated task 15
[Producer] Generated task 16
    Finished task 0
[Consumer 4] Picked task 4
    Processing task 4 (takes 0.64s)
[Producer] Generated task 17
    Finished task 2
[Consumer 1] Picked task 5
    Processing task 5 (takes 1.75s)
[Producer] Generated task 18
    Finished task 3
[